# SFT + GRPO on GSM8K (Qwen3-0.6B)

Before running:
- Set **Accelerator → GPU T4 x2** in the right panel
- Set **Internet → On**

Then click **Save Version → Save & Run All (Full Notebook)**. Safe to close your computer.

In [ ]:
# ── 1. Clone repo ─────────────────────────────────────────────────────────
import shutil, os, sys, subprocess

os.chdir('/kaggle/working')
shutil.rmtree('/kaggle/working/Fine-tuning-and-GRPO-on-Qwen', ignore_errors=True)
subprocess.run(['git', 'clone',
    'https://github.com/202520030411/Fine-tuning-and-GRPO-on-Qwen.git',
    '/kaggle/working/Fine-tuning-and-GRPO-on-Qwen'], check=True)

REPO = '/kaggle/working/Fine-tuning-and-GRPO-on-Qwen'
os.chdir(REPO)
sys.path.insert(0, REPO)
os.environ['HF_HOME']            = '/kaggle/working/.hf'
os.environ['HF_DATASETS_CACHE']  = '/kaggle/working/.hf/datasets'
os.environ['TRANSFORMERS_CACHE'] = '/kaggle/working/.hf/transformers'
print('Repo ready:', os.listdir('scripts'))

In [ ]:
# ── 2. Install dependencies ───────────────────────────────────────────────
import subprocess
subprocess.run(['pip', 'install', '-q',
    'transformers>=4.51.0', 'peft>=0.11.1', 'datasets>=2.19.0',
    'accelerate', 'tqdm', 'typer', 'sentencepiece', 'safetensors',
    'matplotlib', 'trl>=0.12.0', 'tiktoken'
], check=True)
print('Done')

In [ ]:
# ── 3. Download Qwen3-0.6B-Base ───────────────────────────────────────────
from transformers import AutoModelForCausalLM, AutoTokenizer
AutoTokenizer.from_pretrained('Qwen/Qwen3-0.6B-Base')
AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-0.6B-Base')
print('Model cached')

In [ ]:
# ── 4. Prepare datasets (GSM8K, SVAMP, MMLU) ─────────────────────────────
!python -u scripts/prepare_gsm8k.py
!python -u scripts/prepare_svamp.py --limit 500
# MMLU is downloaded on-the-fly during eval (no prep script needed)

In [ ]:
# ── 5. Verify GPU ─────────────────────────────────────────────────────────
import torch
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name}  {p.total_memory // 1024**2} MB')

In [ ]:
# ── 5b. Smoke test (optional) ─────────────────────────────────────
# Skipped by default. To run: python -u scripts/smoke_test.py --keep-tmp
pass


In [ ]:
import subprocess
# ── 6. DoRA-SFT training (~35 min on T4) — main pipeline ────────────
# Uses train_sft.py with --use-dora flag. Same script as LoRA-SFT,
# same hyperparams — only difference is use_dora=True in LoraConfig.
import os
if os.path.exists('/kaggle/working/model/sft_gsm8k/adapter_model.safetensors'):
    print('DoRA-SFT model already exists — skipping training.')
else:
    subprocess.run("""python -u scripts/train_sft.py \\
        --model-name-or-path Qwen/Qwen3-0.6B-Base \\
        --train-path dataset/processed/gsm8k_train.jsonl \\
        --output-dir /kaggle/working/model/sft_gsm8k \\
        --train-log-path /kaggle/working/model/sft_gsm8k/train_log.jsonl \\
        --max-steps 300 \\
        --per-device-batch-size 2 \\
        --grad-accum 16 \\
        --max-length 384 \\
        --fp16 \\
        --gradient-checkpointing \\
        --use-dora \\
        --log-every 10""", shell=True)


In [ ]:
import subprocess
# ── 7. Eval base model ────────────────────────────────────────────────────
import os
if os.path.exists('/kaggle/working/model/eval_base.jsonl'):
    print('Base eval already exists — skipping.')
else:
    subprocess.run("""python -u scripts/eval.py \
        --base-model Qwen/Qwen3-0.6B-Base \
        --test-path dataset/processed/gsm8k_test.jsonl \
        --output-path /kaggle/working/model/eval_base.jsonl \
        --max-new-tokens 256 \
        --batch-size 8 \
        --max-examples 500""", shell=True)

In [ ]:
import subprocess
# ── 8. Eval SFT model ─────────────────────────────────────────────────────
import os
if os.path.exists('/kaggle/working/model/eval_sft.jsonl'):
    print('SFT eval already exists — skipping.')
else:
    subprocess.run("""python -u scripts/eval.py \
        --base-model Qwen/Qwen3-0.6B-Base \
        --adapter-path /kaggle/working/model/sft_gsm8k \
        --test-path dataset/processed/gsm8k_test.jsonl \
        --output-path /kaggle/working/model/eval_sft.jsonl \
        --max-new-tokens 256 \
        --batch-size 8 \
        --max-examples 500""", shell=True)

In [ ]:
# ── 9. GRPO training (~2.5 hrs on T4) ────────────────────────────────────
import subprocess, sys, os
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'trl>=0.12.0', 'tiktoken', 'peft>=0.11.1',
    'transformers>=4.51.0', 'accelerate'], check=True)

# Skip if a fully merged GRPO model already exists
if os.path.exists('/kaggle/working/model/grpo_gsm8k/config.json'):
    print('GRPO model already exists — skipping training.')
else:
    subprocess.run("""python -u scripts/train_grpo.py \
        --model-name-or-path Qwen/Qwen3-0.6B-Base \
        --ref-model-path /kaggle/working/model/sft_gsm8k \
        --train-path dataset/processed/gsm8k_train.jsonl \
        --output-dir /kaggle/working/model/grpo_gsm8k \
        --max-steps 200 \
        --group-size 8 \
        --kl-coef 0.01""", shell=True)

In [ ]:
import subprocess
# ── 10. Eval GRPO model ───────────────────────────────────────────────────
# GRPO is saved as a fully merged model (base + SFT + GRPO).
import os
if os.path.exists('/kaggle/working/model/eval_grpo.jsonl'):
    print('GRPO eval already exists — skipping.')
else:
    subprocess.run("""python -u scripts/eval.py \
        --base-model /kaggle/working/model/grpo_gsm8k \
        --test-path dataset/processed/gsm8k_test.jsonl \
        --output-path /kaggle/working/model/eval_grpo.jsonl \
        --max-new-tokens 256 \
        --batch-size 8 \
        --max-examples 500""", shell=True)

In [ ]:
import subprocess
# ── 11. SVAMP eval (math generalisation) ─────────────────────────────────
import os

if os.path.exists('/kaggle/working/model/eval_svamp_base.jsonl'):
    print('SVAMP base eval already exists — skipping.')
else:
    subprocess.run("""python -u scripts/eval.py \
        --base-model Qwen/Qwen3-0.6B-Base \
        --test-path dataset/processed/svamp_test.jsonl \
        --output-path /kaggle/working/model/eval_svamp_base.jsonl \
        --max-new-tokens 256 --batch-size 8 --max-examples 500""", shell=True)

if os.path.exists('/kaggle/working/model/eval_svamp_sft.jsonl'):
    print('SVAMP SFT eval already exists — skipping.')
else:
    subprocess.run("""python -u scripts/eval.py \
        --base-model Qwen/Qwen3-0.6B-Base \
        --adapter-path /kaggle/working/model/sft_gsm8k \
        --test-path dataset/processed/svamp_test.jsonl \
        --output-path /kaggle/working/model/eval_svamp_sft.jsonl \
        --max-new-tokens 256 --batch-size 8 --max-examples 500""", shell=True)

if os.path.exists('/kaggle/working/model/eval_svamp_grpo.jsonl'):
    print('SVAMP GRPO eval already exists — skipping.')
else:
    subprocess.run("""python -u scripts/eval.py \
        --base-model /kaggle/working/model/grpo_gsm8k \
        --test-path dataset/processed/svamp_test.jsonl \
        --output-path /kaggle/working/model/eval_svamp_grpo.jsonl \
        --max-new-tokens 256 --batch-size 8 --max-examples 500""", shell=True)

In [ ]:
import subprocess
# ── 12. MMLU eval (forgetting check) ─────────────────────────────────────
import os
SUBJ = "high_school_mathematics,elementary_mathematics,world_history"

if os.path.exists('/kaggle/working/model/eval_mmlu_base.jsonl'):
    print('MMLU base eval already exists — skipping.')
else:
    subprocess.run(f"""python -u scripts/eval_mmlu.py \
        --base-model Qwen/Qwen3-0.6B-Base \
        --output-path /kaggle/working/model/eval_mmlu_base.jsonl \
        --subjects "{SUBJ}" --max-examples-per-subject 150 --batch-size 8""", shell=True)

if os.path.exists('/kaggle/working/model/eval_mmlu_sft.jsonl'):
    print('MMLU SFT eval already exists — skipping.')
else:
    subprocess.run(f"""python -u scripts/eval_mmlu.py \
        --base-model Qwen/Qwen3-0.6B-Base \
        --adapter-path /kaggle/working/model/sft_gsm8k \
        --output-path /kaggle/working/model/eval_mmlu_sft.jsonl \
        --subjects "{SUBJ}" --max-examples-per-subject 150 --batch-size 8""", shell=True)

if os.path.exists('/kaggle/working/model/eval_mmlu_grpo.jsonl'):
    print('MMLU GRPO eval already exists — skipping.')
else:
    subprocess.run(f"""python -u scripts/eval_mmlu.py \
        --base-model /kaggle/working/model/grpo_gsm8k \
        --output-path /kaggle/working/model/eval_mmlu_grpo.jsonl \
        --subjects "{SUBJ}" --max-examples-per-subject 150 --batch-size 8""", shell=True)

## Additional Experiments: LoRA-SFT Comparison / Prompt Design / Reasoning Validity

In [ ]:
import subprocess
# ── 14. LoRA-SFT training (DoRA vs LoRA comparison baseline) ─────────
# Same hyperparams as DoRA-SFT (cell 6) but uses standard LoRA.
# Results are compared against DoRA-SFT in the analyze step.
import os
if os.path.exists('/kaggle/working/model/lora_sft_gsm8k/adapter_model.safetensors'):
    print('LoRA-SFT model already exists — skipping training.')
else:
    subprocess.run("""python -u scripts/train_sft.py \\
        --model-name-or-path Qwen/Qwen3-0.6B-Base \\
        --train-path dataset/processed/gsm8k_train.jsonl \\
        --output-dir /kaggle/working/model/lora_sft_gsm8k \\
        --train-log-path /kaggle/working/model/lora_sft_gsm8k/train_log.jsonl \\
        --max-steps 300 \\
        --per-device-batch-size 2 \\
        --grad-accum 16 \\
        --max-length 384 \\
        --fp16 \\
        --gradient-checkpointing \\
        --log-every 10""", shell=True)

# Evaluate LoRA-SFT on GSM8K
if os.path.exists('/kaggle/working/model/eval_lora_sft.jsonl'):
    print('LoRA-SFT GSM8K eval already exists — skipping.')
else:
    subprocess.run("""python -u scripts/eval.py \\
        --base-model Qwen/Qwen3-0.6B-Base \\
        --adapter-path /kaggle/working/model/lora_sft_gsm8k \\
        --test-path dataset/processed/gsm8k_test.jsonl \\
        --output-path /kaggle/working/model/eval_lora_sft.jsonl \\
        --max-new-tokens 256 \\
        --batch-size 8 \\
        --max-examples 500""", shell=True)

# Evaluate LoRA-SFT on SVAMP (generalization check)
if os.path.exists('/kaggle/working/model/eval_svamp_lora_sft.jsonl'):
    print('LoRA-SFT SVAMP eval already exists — skipping.')
else:
    subprocess.run("""python -u scripts/eval.py \\
        --base-model Qwen/Qwen3-0.6B-Base \\
        --adapter-path /kaggle/working/model/lora_sft_gsm8k \\
        --test-path dataset/processed/svamp_test.jsonl \\
        --output-path /kaggle/working/model/eval_svamp_lora_sft.jsonl \\
        --max-new-tokens 256 \\
        --batch-size 8 \\
        --max-examples 500""", shell=True)


In [ ]:
import subprocess
# ── 15. Prompt design comparison (direct vs CoT vs math rules) ────────
import os

if os.path.exists('/kaggle/working/model/prompt_base/prompt_comparison_summary.json'):
    print('Prompt base eval already exists — skipping.')
else:
    subprocess.run("""python -u scripts/eval_prompts.py \\
        --test-path dataset/processed/gsm8k_test.jsonl \\
        --base-model Qwen/Qwen3-0.6B-Base \\
        --output-dir /kaggle/working/model/prompt_base \\
        --max-examples 200""", shell=True)

if os.path.exists('/kaggle/working/model/prompt_sft/prompt_comparison_summary.json'):
    print('Prompt SFT eval already exists — skipping.')
else:
    subprocess.run("""python -u scripts/eval_prompts.py \\
        --test-path dataset/processed/gsm8k_test.jsonl \\
        --base-model Qwen/Qwen3-0.6B-Base \\
        --adapter-path /kaggle/working/model/sft_gsm8k \\
        --output-dir /kaggle/working/model/prompt_sft \\
        --max-examples 200""", shell=True)


In [ ]:
import subprocess
# ── 16. Step-by-step reasoning validity ─────────────────────────
import os

for tag, path in [
    ('base',  '/kaggle/working/model/eval_base.jsonl'),
    ('sft',   '/kaggle/working/model/eval_sft.jsonl'),
    ('grpo',  '/kaggle/working/model/eval_grpo.jsonl'),
]:
    out_dir = f'/kaggle/working/model/reasoning_{tag}'
    sentinel = os.path.join(out_dir, 'reasoning_summary.json')
    if not os.path.exists(path):
        print(f'{tag} eval not found — skipping reasoning analysis.')
    elif os.path.exists(sentinel):
        print(f'Reasoning {tag} already exists — skipping.')
    else:
        subprocess.run(f'python -u scripts/eval_reasoning.py --results-path {path} --output-dir {out_dir}', shell=True)


In [ ]:
# ── 13. Analyze all results + plots ──────────────────────────────────
# --sft-results      = DoRA-SFT (main pipeline, cell 6)
# --lora-sft-results = LoRA-SFT (comparison baseline, cell 14)
# --sft-log          = DoRA-SFT training curve
# --lora-sft-log     = LoRA-SFT training curve (for comparison)
!python -u scripts/analyze.py \\
    --base-results         /kaggle/working/model/eval_base.jsonl \\
    --lora-sft-results     /kaggle/working/model/eval_lora_sft.jsonl \\
    --sft-results          /kaggle/working/model/eval_sft.jsonl \\
    --grpo-results         /kaggle/working/model/eval_grpo.jsonl \\
    --svamp-base           /kaggle/working/model/eval_svamp_base.jsonl \\
    --svamp-sft            /kaggle/working/model/eval_svamp_sft.jsonl \\
    --svamp-grpo           /kaggle/working/model/eval_svamp_grpo.jsonl \\
    --mmlu-base            /kaggle/working/model/eval_mmlu_base.jsonl \\
    --mmlu-sft             /kaggle/working/model/eval_mmlu_sft.jsonl \\
    --mmlu-grpo            /kaggle/working/model/eval_mmlu_grpo.jsonl \\
    --sft-log              /kaggle/working/model/sft_gsm8k/train_log.jsonl \\
    --lora-sft-log         /kaggle/working/model/lora_sft_gsm8k/train_log.jsonl \\
    --grpo-log             /kaggle/working/model/grpo_gsm8k/trainer_state.json \\
    --prompt-base-dir      /kaggle/working/model/prompt_base \\
    --prompt-sft-dir       /kaggle/working/model/prompt_sft \\
    --reasoning-base-dir   /kaggle/working/model/reasoning_base \\
    --reasoning-sft-dir    /kaggle/working/model/reasoning_sft \\
    --reasoning-grpo-dir   /kaggle/working/model/reasoning_grpo \\
    --images-dir           /kaggle/working/images


In [ ]:
# ── 17. Zip everything for download ──────────────────────────────────────
import shutil
shutil.make_archive('/kaggle/working/results', 'zip', '/kaggle/working/model')
shutil.make_archive('/kaggle/working/plots',   'zip', '/kaggle/working/images')
print('ALL DONE — download results.zip and plots.zip from the Output panel')